# Run everything on Colab

This is the only notebook you need to open to run the experiments. It pulls the latest
code from GitHub, so there is nothing to copy between places.

1. Runtime > Change runtime type > **GPU** (T4 works, L4 or A100 is faster)
2. Run the cells from top to bottom
3. Copy the summary printed by the last cell and paste it back in the chat

Section A only re-scores the checkpoints that are already on Drive (about 30 min).
Section B retrains the ViT with the v2 preprocessing (a few hours per seed), so it is off by default.

In [ ]:
# we mount Drive (where the NHATS data and the checkpoints are) and get the latest code
from google.colab import drive
drive.mount('/content/drive')

import os
REPO = '/content/cdt-alzheimer-screening'
if not os.path.exists(REPO):
    !git clone -q https://github.com/wiambenadder/cdt-alzheimer-screening.git {REPO}
else:
    !git -C {REPO} pull -q
%cd {REPO}
!pip install -q -r requirements.txt
!git log --oneline -1

import torch
print('GPU:', torch.cuda.get_device_name(0) if torch.cuda.is_available() else 'NONE, switch the runtime to GPU')

In [ ]:
# the settings, change these if your Drive folder is somewhere else
DRIVE_DATA = '/content/drive/MyDrive/cdt-data'

RUN_SECTION_A = True    # re-score the v1 checkpoints with CIs (no training)
RUN_SECTION_B = False   # retrain ViT-B/16 with v2 preprocessing (long)
B_SEEDS = '42 43 44'

os.environ['CDT_DATA_DIR'] = DRIVE_DATA                       # splits.csv is saved here
os.environ['CDT_MODELS_DIR'] = f'{DRIVE_DATA}/models'
os.environ['CDT_RESULTS_DIR'] = f'{DRIVE_DATA}/results_v2'    # on Drive so nothing is lost
os.makedirs(os.environ['CDT_RESULTS_DIR'], exist_ok=True)
print('checkpoints on Drive:', sorted(os.listdir(os.environ['CDT_MODELS_DIR'])))

In [ ]:
# we copy the TIFFs to the local disk first, reading 59k files from Drive during training is too slow
import shutil, time
import pandas as pd
from pathlib import Path

LOCAL = Path('/content/nhats_local')
LOCAL.mkdir(exist_ok=True)
t0 = time.time()
for rf in sorted(Path(f'{DRIVE_DATA}/nhats_raw').iterdir()):
    if not rf.is_dir():
        continue
    dst = LOCAL / rf.name
    if dst.exists() and any(dst.glob('*.tif')):
        continue
    print(f'[copy] {rf.name}')
    shutil.copytree(rf, dst, dirs_exist_ok=True)
print(f'staged in {(time.time() - t0) / 60:.1f} min')

labels = pd.read_csv(f'{DRIVE_DATA}/labels.csv')
labels['image_path'] = labels['image_path'].str.replace(f'{DRIVE_DATA}/nhats_raw/', str(LOCAL) + '/', regex=False)
LABELS = '/content/labels_local.csv'
labels.to_csv(LABELS, index=False)
print(f'{len(labels):,} labels -> {LABELS}')

## A. Re-score the v1 checkpoints

The v1 models were trained on the v1 crop, so we score them with `--crop-method projection`.
The first run also writes `splits.csv` to Drive, every run after that reuses it.

In [ ]:
if RUN_SECTION_A:
    v1 = {'vit_b16': 'vit_b16_ft', 'vgg16': 'vgg16_ft', 'efficientnet_b0': 'effb0_ft'}
    for model_name, ckpt_name in v1.items():
        ckpt = f"{os.environ['CDT_MODELS_DIR']}/{ckpt_name}_best.pt"
        if not os.path.exists(ckpt):
            print(f'[skip] {ckpt} not found')
            continue
        !python -m src.run --models {model_name} --evaluate-only --crop-method projection \
            --checkpoint {ckpt} --run-tag _v1 --labels {LABELS} --num-workers 4
    !python -m src.run --compare vit_b16_v1_s42 vgg16_v1_s42

## B. Retrain ViT-B/16 with the v2 preprocessing

Turn on `RUN_SECTION_B` in the settings cell first. Each seed takes a few hours,
the checkpoints and results go to Drive so a disconnect doesn't lose finished seeds.

In [ ]:
if RUN_SECTION_B:
    !python -m src.run --models vit_b16 --seeds {B_SEEDS} --select-metric val_kappa \
        --run-tag _v2 --labels {LABELS} --num-workers 4
    if os.path.exists(f"{os.environ['CDT_RESULTS_DIR']}/vit_b16_v1_s42"):
        !python -m src.run --compare vit_b16_v2_s42 vit_b16_v1_s42

## Summary to paste back in the chat

Only aggregate numbers, no participant IDs, so it is fine to share.

In [ ]:
import json, glob
rows = []
for f in sorted(glob.glob(f"{os.environ['CDT_RESULTS_DIR']}/*/metrics.json")):
    m = json.load(open(f))
    k, ci = m['argmax']['quadratic_kappa'], m['argmax_ci']['quadratic_kappa']
    kt, cit = m['tuned']['quadratic_kappa'], m['tuned_ci']['quadratic_kappa']
    rows.append(f"{m['run_name']:24s} n_test={m['n_test']}  kappa={k:.3f} [{ci['lo']:.3f}, {ci['hi']:.3f}]  "
                f"tuned={kt:.3f} [{cit['lo']:.3f}, {cit['hi']:.3f}]  acc={m['argmax']['accuracy']:.3f}  "
                f"macroF1={m['argmax']['macro_f1']:.3f}")
for f in sorted(glob.glob(f"{os.environ['CDT_RESULTS_DIR']}/compare_*.json")):
    c = json.load(open(f))
    rows.append(f"{Path(f).stem}: diff={c['diff']:+.3f} [{c['diff_lo']:+.3f}, {c['diff_hi']:+.3f}]")
print('\n'.join(rows) if rows else 'no results yet')
# the v1 test set had 8,938 clocks, if n_test is different the split is not the same as v1